# Install the necessary packages

In [ ]:
# %pip install 'crewai[tools]' setuptools crewai python-dotenv distutils

In [5]:
#This imports openai api key and serper api key as env variables
from dotenv import load_dotenv
load_dotenv()


True

# Import Necessary packages

In [1]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

/Users/kranthivardhankurumindla/Documents/Agents_Crewai/kranthi09/.venv/lib/python3.12/site-packages/pydantic/_internal/_config.py:295: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
/Users/kranthivardhankurumindla/Documents/Agents_Crewai/kranthi09/.venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "schema" in "DatabricksQueryToolSchema" shadows an attribute in parent "BaseModel"
  warnings.warn(
/Users/kranthivardhankurumindla/Documents/Agents_Crewai/kranthi09/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: <built-in function callable> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even e

# Import External Tools

In [2]:
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

# Definition of Agents
Agent is an autonomous unit that can:
    Perform specific tasks
    Make decisions based on its role and goal
    Use tools to accomplish objectives
    Communicate and collaborate with other agents
    Maintain memory of interactions
    Delegate tasks when allowed

In [3]:
# Healthcare Agents

patient_intake = Agent(
    role='Patient Intake Specialist',
    goal='Gather patient information and medical history, with human verification.',
    backstory="You are a meticulous intake specialist, ensuring accurate patient records. You understand that human verification is crucial for sensitive data.",
    verbose=True,
    allow_delegation=True
)
# Comments:
# - This agent is designed for initial patient interaction.
# - "Human verification" is a critical feature, emphasizing the need for oversight.
# - Delegation allows this agent to pass information to other specialists.
# - Backstory helps define the agent's personality and approach.

diagnostic_expert = Agent(
    role='Diagnostic Expert',
    goal='Analyze patient symptoms and medical history to suggest potential diagnoses, with human verification and external search.',
    backstory="You are a seasoned diagnostician with extensive medical knowledge. You understand that in complex cases human validation is needed. You also utilize external search and website scraping to find the latest advancements.",
    verbose=True,
    allow_delegation=True,
    tools=[search_tool, scrape_tool]  # Add search and scrape tools
)
# Comments:
# - This agent performs the core diagnostic function.
# - "Human verification" is again vital for confirming diagnoses.
# - Integration of search and scraping tools allows for up-to-date information.
# - Delegation supports a collaborative diagnostic process.
# - Backstory establishes expertise and awareness of limitations.

treatment_planner = Agent(
    role='Treatment Planner',
    goal='Develop personalized treatment plans based on diagnoses, with human approval and external search.',
    backstory="You are a skilled treatment planner, focusing on effective and patient-centered care. You will ask for human approval before finalizing the treatment plan. You also utilize external search and website scraping to find the latest advancements.",
    verbose=True,
    allow_delegation=True,
    tools=[search_tool, scrape_tool]  # Add search and scrape tools
)
# Comments:
# - This agent focuses on creating treatment plans based on diagnoses.
# - "Human approval" ensures that plans are reviewed and validated.
# - Use of search and scraping tools allows for the latest treatment options.
# - Delegation enables collaboration with other specialists.
# - Backstory emphasizes patient-centered care.

pharmacy_advisor = Agent(
    role='Pharmacy Advisor',
    goal='Provide information on medications, dosages, and potential interactions.',
    backstory="You are a knowledgeable pharmacy advisor, ensuring safe medication practices.",
    verbose=True,
    allow_delegation=False
)
# Comments:
# - This agent provides specific medication-related information.
# - "Allow delegation=False" suggests that this agent primarily acts as a resource.
# - Backstory highlights the agent's focus on safety.

wellness_coach = Agent(
    role='Wellness Coach',
    goal='Advise on lifestyle changes and wellness strategies to support patient health.',
    backstory="You are a compassionate wellness coach, promoting holistic health.",
    verbose=True,
    allow_delegation=False
)
# Comments:
# - This agent focuses on preventative care and lifestyle improvements.
# - "Allow delegation=False" implies a direct interaction with the patient or other agents that have already collected information.
# - Backstory emphasizes a compassionate and holistic approach.

medical_record_keeper = Agent(
    role='Medical Record Keeper',
    goal='Maintain and update patient medical records with accuracy and confidentiality.',
    backstory="You are a meticulous medical record keeper, ensuring data integrity.",
    verbose=True,
    allow_delegation=False
)
# Comments:
# - This agent manages patient records, ensuring accuracy and privacy.
# - "Allow delegation=False" suggests a focused role in data management.
# - Backstory highlights the agent's focus on data integrity.

# Definition of Tasks
In the CrewAI framework, a Task is a specific assignment completed by an Agent.

Tasks provide all necessary details for execution, such as a description, the agent responsible, required tools, and more, facilitating a wide range of action complexities.

Tasks within CrewAI can be collaborative, requiring multiple agents to work together. This is managed through the task properties and orchestrated by the Crew’s process, enhancing teamwork and efficiency.

In [4]:
# Healthcare Tasks

intake_task = Task(
    description="Gather patient information, including symptoms, medical history, and current medications from the given info {input}.Do not make any assumptions stick to the data given by the user. After gathering the information, present it for human verification. If the human input is 'OK' continue, if it is not, correct the information.",
    agent=patient_intake,
    expected_output="Detailed, human-verified patient information and medical history.",
    human_input=True
)

diagnostic_task = Task(
    description="Analyze the patient's symptoms and medical history to suggest potential diagnoses. Provide a list of possible conditions and relevant tests. If any disease is rare, or the symptoms are complex, ask for a human verification by presenting the gathered information, and potential diagnoses.",
    agent=diagnostic_expert,
    expected_output="A list of potential diagnoses and recommended tests, verified by a human when needed.",
    human_input=True #This will be triggered if the agent determines it is needed.
)

treatment_task = Task(
    description="Based on the diagnostic information, develop a personalized treatment plan, including medications, therapies, and follow-up appointments. Provide the medications needed to the pharmacy advisor. Once you have a treatment plan, present it for human approval, and wait for human input. Only continue if the human input is 'Approved'.",
    agent=treatment_planner,
    expected_output="A comprehensive, human-approved treatment plan.",
    human_input=True
)

pharmacy_task = Task(
    description="Provide information on the medications prescribed in the treatment plan, including dosages, potential side effects, and drug interactions. Send the safe medication usage to the treatment planner and wellness coach.",
    agent=pharmacy_advisor,
    expected_output="Medication information and safety guidelines."
)

wellness_task = Task(
    description="Based on the treatment plan and patient history, provide lifestyle and wellness advice to support the patient's recovery and overall health. Incorporate the safe medication usage from the pharmacy advisor.",
    agent=wellness_coach,
    expected_output="Personalized wellness and lifestyle recommendations."
)

record_keeping_task = Task(
    description="Update the patient's medical records with all information gathered and generated during the process, ensuring accuracy and confidentiality. Summarize all steps for the patient intake.",
    agent=medical_record_keeper,
    expected_output="Updated medical records and a summary of the patient's healthcare journey."
)


# Defining the Crew

A crew in crewAI represents a collaborative group of agents working together to achieve a set of tasks. Each crew defines the strategy for task execution, agent collaboration, and the overall workflow.

In [5]:

# Healthcare Crew

healthcare_crew = Crew(
    agents=[patient_intake, diagnostic_expert, treatment_planner, pharmacy_advisor, wellness_coach, medical_record_keeper],
    tasks=[intake_task, diagnostic_task, treatment_task, pharmacy_task, wellness_task, record_keeping_task],
    verbose=True,
    process=Process.sequential,# Tasks are executed sequentially
    planning=True,
    planning_llm='gpt-4o'  
)
#

# Kickoff the Crew

In [6]:
#Kickoff the healthcare crew
patient_data = "Patient: John Doe, 45 years old. Symptoms: Persistent cough, fever, fatigue. Medical history: Asthma, allergies. Current medications: Albuterol inhaler."

result = healthcare_crew.kickoff(inputs={"input": patient_data})


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9751ff4e-76b7-45ce-b36b-5bd124bcc0c8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 
[2025-03-21 11:49:08][INFO]: Planning the crew execution


🚀 Crew: crew
└── 📋 Task: 4d8f564f-a24c-44e3-88ac-bfda859efcc7
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 4d8f564f-a24c-44e3-88ac-bfda859efcc7
       Status: Executing Task...
    └── 🤖 Agent: Task Execution Planner
            Status: In Progress

🚀 Crew: crew
└── 📋 Task: 4d8f564f-a24c-44e3-88ac-bfda859efcc7
       Status: Executing Task...
    └── 🤖 Agent: Task Execution Planner
            Status: In Progress
        └── 🧠 Thinking...

...

🚀 Crew: crew
├── 📋 Task: 4d8f564f-a24c-44e3-88ac-bfda859efcc7
│      Assigned to: Task Execution Planner
│      Status: ✅ Completed
│   └── 🤖 Agent: Task Execution Planner
│           Status: ✅ Completed
├── 📋 Task: 766e4583-66ca-4659-b16d-720c38715f9d
│      Assigned to: Patient Intake Specialist
│      Status: ✅ Completed
│   └── 🤖 Agent: Patient Intake Specialist
│           Status: ✅ Completed
├── 📋 Task: a13a1117-9c1d-4524-bb70-9b04bc5e317d
│      Assigned to: Diagnostic Expert
│      Status: ✅ Completed
│   ├── 🤖 Agent: Diagnostic Expert
│   │       Status: In Progress
│   │   ├── 🔧 Used Search the internet with Serper (1)
│   │   └── 🔧 Using Delegate work to coworker (2)
│   └── 🤖 Agent: Diagnostic Expert
│           Status: ✅ Completed
├── 📋 Task: efa2438b-8da8-4f2a-b7f8-a3aa856dc0b8
│      Assigned to: Treatment Planner
│      Status: ✅ Completed
│   ├── 🤖 Agent: Treatment Planner
│   │       Status: In Progress
│   │   └── 🔧 Using Delegate work to coworker (3)
│   ├── 🤖 Agent: Patient Intake Specialist
│   │       Status: ✅ Completed
│   │   └── 🔧 Using Delegate work to coworker (4)
│   ├── 🤖 Agent: Patient Intake Specialist
│   │       Status: ✅ Completed
│   │   ├── 🔧 Used Search the internet with Serper (2)
│   │   ├── 🔧 Used Read website content (3)
│   │   └── 🔧 Using Delegate work to coworker (5)
│   └── 🤖 Agent: Treatment Planner
│           Status: ✅ Completed
├── 📋 Task: f4c06cfa-60b5-4392-900c-b5230dc9c973
│      Assigned to: Pharmacy Advisor
│      Status: ✅ Completed
│   └── 🤖 Agent: Pharmacy Advisor
│           Status: ✅ Completed
├── 📋 Task: 95d3781d-9b4d-4427-9638-d8a875401f05
│      Assigned to: Wellness Coach
│      Status: ✅ Completed
│   └── 🤖 Agent: Wellness Coach
│           Status: ✅ Completed
└── 📋 Task: 875ff673-8233-4d71-ab1c-21016fe77ee8
       Status: Executing Task...
    └── 🤖 Agent: Medical Record Keeper
            Status: ✅ Completed

🚀 Crew: crew
├── 📋 Task: 4d8f564f-a24c-44e3-88ac-bfda859efcc7
│      Assigned to: Task Execution Planner
│      Status: ✅ Completed
│   └── 🤖 Agent: Task Execution Planner
│           Status: ✅ Completed
├── 📋 Task: 766e4583-66ca-4659-b16d-720c38715f9d
│      Assigned to: Patient Intake Specialist
│      Status: ✅ Completed
│   └── 🤖 Agent: Patient Intake Specialist
│           Status: ✅ Completed
├── 📋 Task: a13a1117-9c1d-4524-bb70-9b04bc5e317d
│      Assigned to: Diagnostic Expert
│      Status: ✅ Completed
│   ├── 🤖 Agent: Diagnostic Expert
│   │       Status: In Progress
│   │   ├── 🔧 Used Search the internet with Serper (1)
│   │   └── 🔧 Using Delegate work to coworker (2)
│   └── 🤖 Agent: Diagnostic Expert
│           Status: ✅ Completed
├── 📋 Task: efa2438b-8da8-4f2a-b7f8-a3aa856dc0b8
│      Assigned to: Treatment Planner
│      Status: ✅ Completed
│   ├── 🤖 Agent: Treatment Planner
│   │       Status: In Progress
│   │   └── 🔧 Using Delegate work to coworker (3)
│   ├── 🤖 Agent: Patient Intake Specialist
│   │       Status: ✅ Completed
│   │   └── 🔧 Using Delegate work to coworker (4)
│   ├── 🤖 Agent: Patient Intake Specialist
│   │       Status: ✅ Completed
│   │   ├── 🔧 Used Search the internet with Serper (2)
│   │   ├── 🔧 Used Read website content (3)
│   │   └── 🔧 Using Delegate work to coworker (5)
│   └── 🤖 Agent: Treatment Planner
│           Status: ✅ Completed
├── 📋 Task: f4c06cfa-60b5-4392-900c-b5230dc9c973
│      Assigned to: Pharmacy Advisor
│      Status: ✅ Completed
│   └── 🤖 Agent: Pharmacy Advisor
│           Status: ✅ Completed
├── 📋 Task: 95d3781d-9b4d-4427-9638-d8a875401f05
│      Assigned to: Wellness Coach
│      Status: ✅ Completed
│   └── 🤖 Agent: Wellness Coach
│           Status: ✅ Completed
└── 📋 Task: 875ff673-8233-4d71-ab1c-21016fe77ee8
       Assigned to: Medical Record Keeper
       Status: ✅ Completed
    └── 🤖 Agent: Medical Record Keeper
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 875ff673-8233-4d71-ab1c-21016fe77ee8                                                                     │
│  Agent: Medical Record Keeper                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9751ff4e-76b7-45ce-b36b-5bd124bcc0c8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Healthcare Process Result:
I have documented and updated John Doe's medical records comprehensively, ensuring accuracy and confidentiality in all entries. Here is the complete content reflecting John Doe's patient intake journey:

---

**Patient Medical Records:**

**Patient Information:**
- Name: John Doe
- Age: 45 years old
- Symptoms: Persistent cough, fever, fatigue
- Medical History: Asthma, allergies
- Current Medications: Albuterol inhaler

**Potential Diagnoses:**
1. Allergic Asthma
2. Chronic Cough related to a respiratory condition
3. Acute Bronchitis
4. Hypersensitivity Pneumonitis
5. Viral infections (e.g., Flu)

**Recommended Diagnostic Tests:**
- Spirometry
- Bronchoprovocation testing
- Chest X-ray
- Sputum Culture
- COVID-19 test

**Treatment Plan:**

1. **Management of Allergic Asthma:**
   - Continue with Albuterol inhaler for immediate relief.
   - Introduce an inhaled corticosteroid (ICS) such as Fluticasone or Budesonide.
   - Consider a once-daily combination of 

In [ ]:

print("\nHealthcare Process Result:")
print(result)